# SUS CKD Layer Compression: 12-Layer Curves for DR-TabPFN (Work 8, SUS)

This notebook measures, at each of the 12 DR-TabPFN layer outputs, how the target-token embeddings spread, how well a linear probe decodes death, and how the gradient of the death probability changes across depth. It runs three builds on the SUS CKD population:

- **Same-year `last_3` 2011, consistent scaling (primary):** the reference-year roles and context rule of the historical Work 2 build, refitted with context-based scaling.
- **Same-year `last_3` 2011, historical scaling (sensitivity):** the same rows with the cached historical scaling. Its layer-11 values reproduce Work 2 on the 2011 rows.
- **Prospective `last_3` r=2011, consistent scaling:** the row sets of the Work 7 archive. Its layer-11 values reproduce the Work 7 geometry summary.

Evaluation rows are the `evaluation`-role rows of input years 2011 to 2013, pooled. The probe and the standardization scale use reference-year fitting rows only; context rows are never used for the probe, the scale or the geometry. The notebook ends with the FLChain curves of the FLChain notebook overlaid on the same layers.

- **Descriptive only:** the notebook reports curves with bootstrap intervals and the model's own AUROC on the same rows. It does not choose a layer.
- **Device:** CPU with 32 threads. The DR-TabPFN fit on this context does not fit the 8 GB GPU (Work 7 out-of-memory log).

## 1. Imports
Libraries used by the steps below. The helper functions are in `layer_compression.py`, `prospective_audit.py`, `flchain_probe.py`, `tcav_geometry.py` and `concept_subgroup_drift_pilot.py`.

In [1]:
import gc
import json
import os
import resource
import subprocess
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import concept_subgroup_drift_pilot as pilot
import flchain_probe as probe
import layer_compression as lc
import prospective_audit as audit
import tcav_geometry as geometry

T0 = time.monotonic()
STARTED_UTC = datetime.now(timezone.utc).isoformat()
STEP_SECONDS = {}

## 2. Constants
One cell holds every setting. The dry run before the full run edits only `EVAL_ROWS_PER_STRATUM` and `BOOTSTRAP_DRAWS`. Seeds: fit and split `42` (as Work 7), bootstrap `20261106`, random directions `20261124` (fixed inside `flchain_probe`). Tolerances are the engineering tolerances of the plan. The expected counts are the row counts recorded by Work 7 and Work 2 and by the plan. The output folder follows `$RPI_OUT_DIR`.

In [2]:
POPULATION_PATH = Path("stats/temporal_robustness/_population/prepared.pkl")
POPULATION_SHA256 = "c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7"
PROSPECTIVE_ARCHIVE_PATH = Path("stats/prospective_target_audit/scope_a/prospective_splits_and_metadata.npz")
PROSPECTIVE_GEOMETRY_PATH = Path("stats/prospective_target_audit/scope_a/geometry_summary.csv")
WORK2_DIR = Path("stats/concept_agreement_study/extensions/tcav_geometry/full")
WORK2_BUILD = "last_3"
FLCHAIN_CURVES_PATH = Path("stats/flchain_layer_compression/layer_curves.csv")
OUT_DIR = Path(os.environ.get("RPI_OUT_DIR", "stats/sus_layer_compression"))
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_YEAR = 2011
EVALUATION_INPUT_YEARS = (2011, 2012, 2013)
CONTEXT_FIRST_YEAR = audit.context_first_year(REFERENCE_YEAR)
FITTING_ROLES = ("sae_discovery", "rule_discovery", "cav_fit")
STRATUM = "input_years_2011_2013"
BUILDS = [
    {"key": "same_year_consistent", "build": "same_year_last_3_2011", "design": "same_year",
     "scaling": "consistent", "use_context_scaler": True, "label": "same-year 2011, consistent"},
    {"key": "same_year_historical", "build": "same_year_last_3_2011", "design": "same_year",
     "scaling": "historical", "use_context_scaler": False, "label": "same-year 2011, historical"},
    {"key": "prospective_consistent", "build": "prospective_last_3_2011", "design": "prospective",
     "scaling": "consistent", "use_context_scaler": True, "label": "prospective r=2011, consistent"},
]
DEVICE = "cpu"
PREDICTOR_THREADS = 32
torch.set_num_threads(PREDICTOR_THREADS)
FIT_SEED = audit.SEED
BOOTSTRAP_SEED = audit.BOOTSTRAP_SEED
DIRECTION_SEED = probe.DIRECTION_SEED
FD_DIRECTION_SEED = 42
BOOTSTRAP_DRAWS = 500
EVAL_ROWS_PER_STRATUM = None
CAPTURE_BATCH_ROWS = 256
REPLAY_BATCH_ROWS = 256
FD_ROWS = 8
LAYER_COUNT = 12
LAST_LAYER = LAYER_COUNT - 1
FORWARD_TOLERANCE = {"rtol": 1e-4, "atol": 1e-5}
GRADIENT_TOLERANCE = {"rtol": 1e-3, "atol": 1e-5}
EXPECTED_PROSPECTIVE = {"context_rows": 5651, "fitting_rows": 696, "fitting_deaths": 35,
                        "scale_rows": 278, "evaluation_rows": 863, "evaluation_deaths": 57}
EXPECTED_SAME_YEAR = {"context_rows": 6303, "context_current_rows": 1260, "context_prior_rows": 5043,
                      "fitting_rows": 789, "fitting_deaths": 81, "scale_rows": 316,
                      "evaluation_rows": 974, "evaluation_deaths": 90,
                      "evaluation_2011_rows": 314, "evaluation_2011_deaths": 32}

checks = []


def record_check(check, measured, rule, passed, build="all", scaling="none"):
    checks.append({"check": check, "build": build, "scaling": scaling, "measured": measured,
                   "rule": rule, "passed": bool(passed)})


def write_checks_and_stop_on_failure():
    checks_table = pd.DataFrame(checks)
    checks_table.to_csv(OUT_DIR / "checks.csv", index=False)
    failed = checks_table.loc[~checks_table["passed"], "check"].tolist()
    if failed:
        raise RuntimeError(f"Engineering checks failed: {failed}")
    return checks_table


print(f"Device: {DEVICE} | threads: {PREDICTOR_THREADS} | Out: {OUT_DIR} | Torch: {torch.__version__}")
print(f"Evaluation rows per stratum: {'all' if EVAL_ROWS_PER_STRATUM is None else EVAL_ROWS_PER_STRATUM}")

Device: cpu | threads: 32 | Out: /home/rafaelgomes/SAE-TCAV/stats/sus_layer_compression | Torch: 2.1.2+cu121
Evaluation rows per stratum: all


## 3. Input Integrity
The population file is checked against its pinned SHA-256 before it is read (`load_population` refuses a different file). The SHA-256 of every other input is recorded here: the Work 7 archive and geometry summary, the Work 2 geometry tables and the FLChain curves used for the overlay.

In [3]:
step_start = time.monotonic()
population = audit.load_population(POPULATION_PATH, POPULATION_SHA256)
population_sha256 = population["fingerprint"]["file_sha256"]
record_check("population file sha256 equals the approved value", population_sha256, POPULATION_SHA256,
             population_sha256 == POPULATION_SHA256)
record_check("population feature count", population["X"].shape[1], "= 500", population["X"].shape[1] == 500)
year = population["year"]
patient_id = population["patient_id"]
X = population["X"]
with np.load(PROSPECTIVE_ARCHIVE_PATH) as archive:
    prospective_archive = {name: archive[name] for name in archive.files}
input_sha256 = {
    "prepared.pkl": population_sha256,
    "prospective_splits_and_metadata.npz": probe.file_sha256(PROSPECTIVE_ARCHIVE_PATH),
    "work7_geometry_summary.csv": probe.file_sha256(PROSPECTIVE_GEOMETRY_PATH),
    "work2_embedding_spectrum.csv": probe.file_sha256(WORK2_DIR / "embedding_spectrum.csv"),
    "work2_coherence.csv": probe.file_sha256(WORK2_DIR / "coherence.csv"),
    "work2_gradient_dimension.csv": probe.file_sha256(WORK2_DIR / "gradient_dimension.csv"),
    "flchain_layer_curves.csv": probe.file_sha256(FLCHAIN_CURVES_PATH),
}
print(f"population rows: {population['fingerprint']['rows']} | patients: {population['fingerprint']['patients']}")
print(input_sha256)
STEP_SECONDS["inputs"] = round(time.monotonic() - step_start, 2)

population rows: 21900 | patients: 5636
{'prepared.pkl': 'c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7', 'prospective_splits_and_metadata.npz': '6f7b580113778a47c4fe1ffae471a922dd2bdd948bb1c32962d32c8f70cd534b', 'work7_geometry_summary.csv': '2c1bbe8d1197b53790b5f9347987ef165e4f20e2609f1ca2fcf01f263d4c915a', 'work2_embedding_spectrum.csv': 'a482d0ac32b9b6a48b89f149b7607ea0bde238950277409e5bd059604023271a', 'work2_coherence.csv': '345b3b016d104e284078a32f7f2ab68bfde61f8d3ed617761f7b396653e2f9a8', 'work2_gradient_dimension.csv': '8e93d5cd174b3bb141b0e78f23bf4cba3c20eda4075c5bd933a5a2ebdffb9d49', 'flchain_layer_curves.csv': 'a915d4dd5c6c3633d66f9acb036659d833d62c306de62fad58613059c611169b'}


## 4. Row Sets
Same-year rows follow the reference-year roles (`pilot.reference_year_roles`) and the context rule of the historical Work 2 build: year-2011 `tabpfn_context` non-entrants plus the 2009 and 2010 rows that are not post-death. Prospective rows come from the Work 7 archive: its context mask, its reference-year roles and its outcome labels (`y_next`). Fitting rows are reference-year rows with the fitting roles; the standardization scale uses the `sae_discovery` rows only. Evaluation rows are the `evaluation` role in input years 2011 to 2013. Counts are measured on the full populations, before any dry-run limit.

In [4]:
step_start = time.monotonic()
roles = pilot.reference_year_roles(population, REFERENCE_YEAR, FIT_SEED)
pilot.check_role_coverage(population, roles)
role = roles["row_role"]
is_entrant = roles["is_entrant"]

same_year_context_current = (year == REFERENCE_YEAR) & (role == "tabpfn_context") & ~is_entrant
same_year_context_prior = (year >= CONTEXT_FIRST_YEAR) & (year < REFERENCE_YEAR) & ~population["post_death"]
same_year_context = same_year_context_current | same_year_context_prior
same_year_fitting = (year == REFERENCE_YEAR) & np.isin(role, FITTING_ROLES)
same_year_scale = (year == REFERENCE_YEAR) & (role == "sae_discovery")
same_year_evaluation = np.isin(year, EVALUATION_INPUT_YEARS) & (role == "evaluation")

in_cohort = prospective_archive["in_cohort"]
outcome_valid = prospective_archive["outcome_valid"]
prospective_role = prospective_archive[f"role_{REFERENCE_YEAR}"]
prospective_context_archive = prospective_archive[f"last_3_context_{REFERENCE_YEAR}"]
cohort_rows = np.flatnonzero(in_cohort)
cohort = pd.DataFrame({
    "year": year[cohort_rows],
    "role": prospective_role[cohort_rows],
    "outcome_valid": outcome_valid[cohort_rows],
    "y_next": prospective_archive["y_next"][cohort_rows],
})
prospective_fitting = ((year == REFERENCE_YEAR) & in_cohort & outcome_valid
                       & np.isin(prospective_role, FITTING_ROLES))
prospective_scale = (year == REFERENCE_YEAR) & in_cohort & (prospective_role == "sae_discovery")
prospective_evaluation = cohort_rows[audit.evaluation_mask(cohort, REFERENCE_YEAR)]

roles_match = bool(np.array_equal(prospective_role, role))
record_check("prospective archive roles equal reference-year roles", int(roles_match),
             "identical role string on every population row", roles_match, build="prospective_last_3_2011")
context_match = bool(np.array_equal(audit.context_mask(cohort, REFERENCE_YEAR),
                                    prospective_context_archive[cohort_rows]))
record_check("prospective context mask equals prospective_audit.context_mask", int(context_match),
             "identical boolean on every cohort row", context_match, build="prospective_last_3_2011")

ROW_SETS = {
    "same_year": {"context": np.flatnonzero(same_year_context), "fit": np.flatnonzero(same_year_fitting),
                  "scale": np.flatnonzero(same_year_scale), "evaluation": np.flatnonzero(same_year_evaluation),
                  "label": population["y"].astype(float)},
    "prospective": {"context": np.flatnonzero(prospective_context_archive), "fit": np.flatnonzero(prospective_fitting),
                    "scale": np.flatnonzero(prospective_scale), "evaluation": prospective_evaluation,
                    "label": prospective_archive["y_next"]},
}

for design, rows in ROW_SETS.items():
    lc.check_disjoint_rows(rows["context"], rows["fit"], rows["evaluation"])
    record_check(f"{design} context, fitting and evaluation rows are disjoint", 0,
                 "check_disjoint_rows: no row in two sets", True)
    scale_inside_fit = bool(np.isin(rows["scale"], rows["fit"]).all())
    record_check(f"{design} scale rows are fitting rows", int(scale_inside_fit),
                 "every scale row is a fitting row", scale_inside_fit)
    labels_finite = bool(np.isfinite(rows["label"][rows["context"]]).all()
                         and np.isfinite(rows["label"][rows["fit"]]).all()
                         and np.isfinite(rows["label"][rows["evaluation"]]).all())
    record_check(f"{design} labels finite on context, fitting and evaluation rows", int(labels_finite),
                 "no missing label", labels_finite)

same_year_rows = ROW_SETS["same_year"]
same_year_2011 = same_year_rows["evaluation"][year[same_year_rows["evaluation"]] == REFERENCE_YEAR]
prospective_rows = ROW_SETS["prospective"]
counts = []
for design, rows in ROW_SETS.items():
    for name in ("context", "fit", "scale", "evaluation"):
        index = rows[name]
        counts.append({"design": design, "set": name, "rows": len(index),
                       "deaths": int(np.sum(rows["label"][index]))})
counts.append({"design": "same_year", "set": "evaluation_2011", "rows": len(same_year_2011),
               "deaths": int(np.sum(same_year_rows["label"][same_year_2011]))})
counts = pd.DataFrame(counts)
print(counts.to_string(index=False))

expected_counts = [
    ("prospective context rows", len(prospective_rows["context"]), EXPECTED_PROSPECTIVE["context_rows"],
     "prospective_last_3_2011"),
    ("prospective fitting rows", len(prospective_rows["fit"]), EXPECTED_PROSPECTIVE["fitting_rows"],
     "prospective_last_3_2011"),
    ("prospective fitting deaths", int(np.sum(prospective_rows["label"][prospective_rows["fit"]])),
     EXPECTED_PROSPECTIVE["fitting_deaths"], "prospective_last_3_2011"),
    ("prospective scale rows", len(prospective_rows["scale"]), EXPECTED_PROSPECTIVE["scale_rows"],
     "prospective_last_3_2011"),
    ("prospective evaluation rows", len(prospective_rows["evaluation"]), EXPECTED_PROSPECTIVE["evaluation_rows"],
     "prospective_last_3_2011"),
    ("prospective evaluation deaths", int(np.sum(prospective_rows["label"][prospective_rows["evaluation"]])),
     EXPECTED_PROSPECTIVE["evaluation_deaths"], "prospective_last_3_2011"),
    ("same-year scale rows", len(same_year_rows["scale"]), EXPECTED_SAME_YEAR["scale_rows"], "same_year_last_3_2011"),
    ("same-year evaluation rows, input years 2011 to 2013", len(same_year_rows["evaluation"]),
     EXPECTED_SAME_YEAR["evaluation_rows"], "same_year_last_3_2011"),
    ("same-year evaluation deaths, input years 2011 to 2013",
     int(np.sum(same_year_rows["label"][same_year_rows["evaluation"]])),
     EXPECTED_SAME_YEAR["evaluation_deaths"], "same_year_last_3_2011"),
    ("same-year evaluation rows, input year 2011", len(same_year_2011),
     EXPECTED_SAME_YEAR["evaluation_2011_rows"], "same_year_last_3_2011"),
    ("same-year evaluation deaths, input year 2011", int(np.sum(same_year_rows["label"][same_year_2011])),
     EXPECTED_SAME_YEAR["evaluation_2011_deaths"], "same_year_last_3_2011"),
    ("same-year context rows, all input years", len(same_year_rows["context"]),
     EXPECTED_SAME_YEAR["context_rows"], "same_year_last_3_2011"),
    ("same-year context rows, input year 2011 (tabpfn_context, non-entrant)", int(np.sum(same_year_context_current)),
     EXPECTED_SAME_YEAR["context_current_rows"], "same_year_last_3_2011"),
    ("same-year context rows, years before 2011 (not post-death)", int(np.sum(same_year_context_prior)),
     EXPECTED_SAME_YEAR["context_prior_rows"], "same_year_last_3_2011"),
    ("same-year fitting rows", len(same_year_rows["fit"]),
     EXPECTED_SAME_YEAR["fitting_rows"], "same_year_last_3_2011"),
    ("same-year fitting deaths", int(np.sum(same_year_rows["label"][same_year_rows["fit"]])),
     EXPECTED_SAME_YEAR["fitting_deaths"], "same_year_last_3_2011"),
]
for check, measured, expected, build_name in expected_counts:
    record_check(check, measured, f"= {expected} (Work 7 archive / Work 2 / plan Decision 6)",
                 measured == expected, build=build_name)

BUILD_ROWS = {}
for build in BUILDS:
    rows = ROW_SETS[build["design"]]
    evaluation = rows["evaluation"] if EVAL_ROWS_PER_STRATUM is None else rows["evaluation"][:EVAL_ROWS_PER_STRATUM]
    BUILD_ROWS[build["key"]] = {
        "context": rows["context"],
        "fit": rows["fit"],
        "scale_in_fit": np.isin(rows["fit"], rows["scale"]),
        "evaluation": evaluation,
        "label": rows["label"],
    }
STEP_SECONDS["row_sets"] = round(time.monotonic() - step_start, 2)

     design             set  rows  deaths
  same_year         context  6303     564
  same_year             fit   789      81
  same_year           scale   316      33
  same_year      evaluation   974      90
prospective         context  5651     420
prospective             fit   696      35
prospective           scale   278      15
prospective      evaluation   863      57
  same_year evaluation_2011   314      32


## 5. Fit the Three Context Models
Each build fits DR-TabPFN on its context rows with `prospective_audit.fit_dr_tabpfn_context` (seed 42, CPU). The fit raises if any context row is subsampled, and the notebook also records the context count. The token count `T` is read from the fitted model, and the identity of the cached model is recorded before any capture.

In [5]:
fits, token_counts, identity_before = {}, {}, {}
for build in BUILDS:
    key = build["key"]
    rows = BUILD_ROWS[key]
    started = time.monotonic()
    fitted = audit.fit_dr_tabpfn_context(
        X[rows["context"]], rows["label"][rows["context"]], year[rows["context"]],
        domain_origin=CONTEXT_FIRST_YEAR, device=DEVICE, seed=FIT_SEED,
        use_context_scaler=build["use_context_scaler"])
    record_check("context rows used without subsampling", fitted["subsample_rows"],
                 f"subsample_rows == context_rows == {len(rows['context'])}",
                 fitted["subsample_rows"] == fitted["context_rows"] == len(rows["context"]),
                 build=build["build"], scaling=build["scaling"])
    fits[key] = fitted
    token_counts[key] = lc.token_count(fitted["classifier"])
    identity_before[key] = pilot.predictor_context_identity(fitted["classifier"])
    STEP_SECONDS[f"fit_{key}"] = round(time.monotonic() - started, 2)
    print(f"{key}: context rows {fitted['context_rows']} | T = {token_counts[key]} | "
          f"checkpoint {fitted['checkpoint_sha256'][:12]} | seconds {STEP_SECONDS[f'fit_{key}']}")

Initialized decoder for standard with (None, 10)  and nout 10


same_year_consistent: context rows 6303 | T = 121 | checkpoint e653d21bd9b2 | seconds 212.55


same_year_historical: context rows 6303 | T = 121 | checkpoint e653d21bd9b2 | seconds 252.61


prospective_consistent: context rows 5651 | T = 104 | checkpoint e653d21bd9b2 | seconds 183.41


## 6. Capture All 12 Layers
One capture call per row set. The fitting rows give the target and feature-mean embeddings for the scale and the probe. The evaluation rows also keep the layer-0 output (the replay start), the death probability and the head continuation.

In [6]:
captures = {}
for build in BUILDS:
    key = build["key"]
    rows = BUILD_ROWS[key]
    started = time.monotonic()
    fit_capture = lc.capture_layers(fits[key], X[rows["fit"]], year[rows["fit"]], batch_size=CAPTURE_BATCH_ROWS)
    evaluation_capture = lc.capture_layers(fits[key], X[rows["evaluation"]], year[rows["evaluation"]],
                                           batch_size=CAPTURE_BATCH_ROWS)
    shapes_ok = (fit_capture["target"].shape == (len(rows["fit"]), LAYER_COUNT, 192)
                 and evaluation_capture["target"].shape == (len(rows["evaluation"]), LAYER_COUNT, 192)
                 and tuple(evaluation_capture["layer0"].shape) == (len(rows["evaluation"]), token_counts[key], 192))
    record_check("capture shapes all layers", LAYER_COUNT,
                 "target and feature mean (rows, 12, 192); layer 0 full (rows, T, 192)", shapes_ok,
                 build=build["build"], scaling=build["scaling"])
    captures[key] = {"fit": fit_capture, "evaluation": evaluation_capture}
    STEP_SECONDS[f"capture_{key}"] = round(time.monotonic() - started, 2)
    print(f"{key}: fitting rows {len(rows['fit'])} | evaluation rows {len(rows['evaluation'])} | "
          f"layers {LAYER_COUNT} | seconds {STEP_SECONDS[f'capture_{key}']}")

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


same_year_consistent: fitting rows 789 | evaluation rows 974 | layers 12 | seconds 90.81


same_year_historical: fitting rows 789 | evaluation rows 974 | layers 12 | seconds 86.15


prospective_consistent: fitting rows 696 | evaluation rows 863 | layers 12 | seconds 63.98


## 7. Replay Layers 1..11 and Take Gradients at Every Layer
The cached model cannot be differentiated, so the layers are recomputed from copies of the weights and the head-0 context keys and values. One backward pass of the summed death probability gives the target-token gradient at every layer. Evaluation rows are processed in batches; rows do not interact, so batch size does not change the result.

In [7]:
weights_by_key, gradients_by_key = {}, {}
for build in BUILDS:
    key = build["key"]
    evaluation_capture = captures[key]["evaluation"]
    started = time.monotonic()
    weights = lc.clone_replay_weights(fits[key]["classifier"])
    gradient_chunks = []
    for start in range(0, len(evaluation_capture["layer0"]), REPLAY_BATCH_ROWS):
        first_layer = evaluation_capture["layer0"][start:start + REPLAY_BATCH_ROWS]
        _, gradients = lc.layer_gradients(weights, evaluation_capture["continuation"], first_layer)
        gradient_chunks.append(torch.stack(gradients, dim=1).numpy())
    weights_by_key[key] = weights
    gradients_by_key[key] = np.concatenate(gradient_chunks)
    STEP_SECONDS[f"replay_gradients_{key}"] = round(time.monotonic() - started, 2)
    print(f"{key}: gradient array {gradients_by_key[key].shape} (rows, layers, 192)")

same_year_consistent: gradient array (974, 12, 192) (rows, layers, 192)


same_year_historical: gradient array (974, 12, 192) (rows, layers, 192)


prospective_consistent: gradient array (863, 12, 192) (rows, layers, 192)


## 8. Replay Parity and Finite-Difference Checks
Per layer, the replayed target token and feature mean must equal the captured ones, and the replayed probability must equal the captured probability. Then a central finite-difference check, in float64 on the first `8` evaluation rows, tests the gradient of every layer. The layer-11 gradient of every evaluation row is also compared with `pilot.batched_death_gradients` (the Work 2 and Work 7 gradient path) at the gradient tolerance.

In [8]:
fd_by_key = {}
for build in BUILDS:
    key = build["key"]
    evaluation_capture = captures[key]["evaluation"]
    weights = weights_by_key[key]
    target_chunks, mean_chunks, probability_chunks = [], [], []
    with torch.no_grad():
        for start in range(0, len(evaluation_capture["layer0"]), REPLAY_BATCH_ROWS):
            first_layer = evaluation_capture["layer0"][start:start + REPLAY_BATCH_ROWS]
            outputs = lc.replay_layers(weights, first_layer, 0)
            target_chunks.append(torch.stack([output[:, -1, :] for output in outputs], dim=1).numpy())
            mean_chunks.append(torch.stack([output[:, :-1, :].mean(dim=1) for output in outputs], dim=1).numpy())
            probability_chunks.append(lc.death_probability(evaluation_capture["continuation"], outputs[-1]).numpy())
    replayed_target = np.concatenate(target_chunks)
    replayed_mean = np.concatenate(mean_chunks)
    replayed_probability = np.concatenate(probability_chunks)

    target_gap = np.abs(replayed_target - evaluation_capture["target"]).max(axis=(0, 2))
    mean_gap = np.abs(replayed_mean - evaluation_capture["feature_mean"]).max(axis=(0, 2))
    parity_table = pd.DataFrame({"layer": range(LAYER_COUNT), "target_max_abs_gap": target_gap,
                                 "feature_mean_max_abs_gap": mean_gap})
    print(parity_table.to_string(index=False))
    layers_match = (np.allclose(replayed_target, evaluation_capture["target"], **FORWARD_TOLERANCE)
                    and np.allclose(replayed_mean, evaluation_capture["feature_mean"], **FORWARD_TOLERANCE))
    record_check("replay matches cached layers", float(max(target_gap.max(), mean_gap.max())),
                 "allclose rtol 1e-4, atol 1e-5, every layer 0..11", layers_match,
                 build=build["build"], scaling=build["scaling"])
    cached_probability = evaluation_capture["probabilities"][:, 1]
    probability_gap = float(np.abs(replayed_probability - cached_probability).max())
    record_check("replay probability matches cached", probability_gap, "allclose rtol 1e-4, atol 1e-5",
                 np.allclose(replayed_probability, cached_probability, **FORWARD_TOLERANCE),
                 build=build["build"], scaling=build["scaling"])

    fd_report = lc.layer_fd_check(weights, evaluation_capture["continuation"], evaluation_capture["layer0"],
                                  rows=np.arange(FD_ROWS), seed=FD_DIRECTION_SEED)
    fd_by_key[key] = [round(value, 9) for value in fd_report["max_abs_error"]]
    record_check("finite difference every layer", max(fd_by_key[key]),
                 "central difference, step 1e-2, float64, allclose rtol 1e-3, atol 1e-5; 8 rows per layer",
                 fd_report["passed"], build=build["build"], scaling=build["scaling"])
    pilot_gradients = pilot.batched_death_gradients(evaluation_capture["continuation"],
                                                    evaluation_capture["target"][:, LAST_LAYER],
                                                    batch_rows=REPLAY_BATCH_ROWS)
    replayed_last_gradients = gradients_by_key[key][:, LAST_LAYER]
    row_passed = np.isclose(replayed_last_gradients, pilot_gradients, **GRADIENT_TOLERANCE).all(axis=1)
    record_check("layer 11 gradient matches pilot.batched_death_gradients",
                 float(np.abs(replayed_last_gradients - pilot_gradients).max()),
                 "every evaluation row, all 192 components, allclose rtol 1e-3, atol 1e-5; measured = max abs difference",
                 bool(row_passed.all()), build=build["build"], scaling=build["scaling"])
    print(key, "| replay parity max gap:", f"{max(target_gap.max(), mean_gap.max()):.2e}",
          "| FD max abs error by layer:", fd_by_key[key])

 layer  target_max_abs_gap  feature_mean_max_abs_gap
     0                 0.0                       0.0
     1                 0.0                       0.0
     2                 0.0                       0.0
     3                 0.0                       0.0
     4                 0.0                       0.0
     5                 0.0                       0.0
     6                 0.0                       0.0
     7                 0.0                       0.0
     8                 0.0                       0.0
     9                 0.0                       0.0
    10                 0.0                       0.0
    11                 0.0                       0.0


same_year_consistent | replay parity max gap: 0.00e+00 | FD max abs error by layer: [1.99e-06, 1.499e-06, 7.11e-07, 1.859e-06, 3.908e-06, 1.257e-06, 1.783e-06, 1.519e-06, 2.427e-06, 1.781e-06, 1.631e-06, 9.05e-07]


 layer  target_max_abs_gap  feature_mean_max_abs_gap
     0                 0.0                       0.0
     1                 0.0                       0.0
     2                 0.0                       0.0
     3                 0.0                       0.0
     4                 0.0                       0.0
     5                 0.0                       0.0
     6                 0.0                       0.0
     7                 0.0                       0.0
     8                 0.0                       0.0
     9                 0.0                       0.0
    10                 0.0                       0.0
    11                 0.0                       0.0


same_year_historical | replay parity max gap: 0.00e+00 | FD max abs error by layer: [1.59e-06, 1.966e-06, 3.828e-06, 1.482e-06, 1.291e-06, 1.639e-06, 8.12e-07, 1.246e-06, 1.429e-06, 2.546e-06, 3.508e-06, 2.16e-06]


 layer  target_max_abs_gap  feature_mean_max_abs_gap
     0                 0.0                       0.0
     1                 0.0                       0.0
     2                 0.0                       0.0
     3                 0.0                       0.0
     4                 0.0                       0.0
     5                 0.0                       0.0
     6                 0.0                       0.0
     7                 0.0                       0.0
     8                 0.0                       0.0
     9                 0.0                       0.0
    10                 0.0                       0.0
    11                 0.0                       0.0


prospective_consistent | replay parity max gap: 0.00e+00 | FD max abs error by layer: [1.737e-06, 1.311e-06, 1.697e-06, 1.508e-06, 1.368e-06, 2.056e-06, 9.32e-07, 1.637e-06, 1.373e-06, 2.218e-06, 2.919e-06, 1.284e-06]


## 9. Model Unchanged After Replay
The cached model must be unchanged by the replay (identity hash of the fitted context and cached decoder parameters). Re-scoring the first evaluation rows of each build must reproduce the captured probabilities.

In [9]:
for build in BUILDS:
    key = build["key"]
    evaluation_capture = captures[key]["evaluation"]
    identity_after = pilot.predictor_context_identity(fits[key]["classifier"])
    identity_unchanged = identity_after == identity_before[key]
    record_check("predictor identity unchanged by replay", int(identity_unchanged),
                 "predictor_context_identity equal before and after replay", identity_unchanged,
                 build=build["build"], scaling=build["scaling"])
    recheck_rows = BUILD_ROWS[key]["evaluation"][:FD_ROWS]
    recheck = audit.capture_scored_rows(fits[key], X[recheck_rows], year[recheck_rows], batch_size=FD_ROWS)
    cached_first = evaluation_capture["probabilities"][:FD_ROWS, 1]
    recheck_gap = float(np.abs(recheck["probabilities"] - cached_first).max())
    record_check("predictions unchanged after replay", recheck_gap,
                 "allclose rtol 1e-4, atol 1e-5 against the captured probabilities",
                 np.allclose(recheck["probabilities"], cached_first, **FORWARD_TOLERANCE),
                 build=build["build"], scaling=build["scaling"])
    print(key, "| identity unchanged:", identity_unchanged, "| re-scoring gap:", f"{recheck_gap:.2e}")

same_year_consistent | identity unchanged: True | re-scoring gap: 0.00e+00


same_year_historical | identity unchanged: True | re-scoring gap: 0.00e+00


prospective_consistent | identity unchanged: True | re-scoring gap: 0.00e+00


## 10. Release the Fitted Models
The small metadata of each build is kept. The fitted models and the replay copies are then released before the geometry and the bootstrap, to limit memory.

In [10]:
model_metadata = {}
for build in BUILDS:
    key = build["key"]
    fitted = fits[key]
    model_metadata[key] = {"context_rows": int(fitted["context_rows"]),
                           "subsample_rows": int(fitted["subsample_rows"]),
                           "checkpoint_sha256": fitted["checkpoint_sha256"],
                           "token_count": token_counts[key]}
fits.clear()
weights_by_key.clear()
gc.collect()
print("released fitted models and replay copies")

released fitted models and replay copies


## 11. Geometry Per Build
The standardization scale of each layer comes from the `sae_discovery` fitting rows of that build. The layer geometry uses the evaluation rows: target token (spectrum, gradient coherence and dimension, random-direction shares) and feature-mean token (spectrum). Zero-SD columns of each scale are counted per layer.

In [11]:
geometry_tables, scale_rows_by_key, layer_scales_by_key, zero_columns_by_key = {}, {}, {}, {}
for build in BUILDS:
    key = build["key"]
    rows = BUILD_ROWS[key]
    fit_capture = captures[key]["fit"]
    evaluation_capture = captures[key]["evaluation"]
    started = time.monotonic()
    scale_mask = rows["scale_in_fit"]
    scale_target = fit_capture["target"][scale_mask]
    scale_feature_mean = fit_capture["feature_mean"][scale_mask]
    scale_rows_by_key[key] = {"target": scale_target, "feature_mean": scale_feature_mean}
    target_scales = [probe.standardization_scale(scale_target[:, layer]) for layer in range(LAYER_COUNT)]
    mean_scales = [probe.standardization_scale(scale_feature_mean[:, layer]) for layer in range(LAYER_COUNT)]
    layer_scales_by_key[key] = {"target": np.stack([scale for scale, _ in target_scales]),
                                "feature_mean": np.stack([scale for scale, _ in mean_scales])}
    zero_columns_by_key[key] = [int(zero) for _, zero in target_scales]
    table = lc.layer_geometry_table(
        evaluation_capture["target"], gradients_by_key[key], evaluation_capture["probabilities"][:, 1],
        scale_target, feature_mean=evaluation_capture["feature_mean"], scale_feature_mean=scale_feature_mean)
    table.insert(0, "stratum", STRATUM)
    table.insert(0, "scaling", build["scaling"])
    table.insert(0, "build", build["build"])
    geometry_tables[key] = table
    STEP_SECONDS[f"geometry_{key}"] = round(time.monotonic() - started, 2)
    print(key, "| scale rows", len(scale_target), "| zero-SD columns by layer", zero_columns_by_key[key])

layer_geometry = pd.concat(geometry_tables.values(), ignore_index=True)
layer_geometry.to_csv(OUT_DIR / "layer_geometry.csv", index=False)

same_year_consistent | scale rows 316 | zero-SD columns by layer [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


same_year_historical | scale rows 316 | zero-SD columns by layer [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


prospective_consistent | scale rows 278 | zero-SD columns by layer [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 12. Layer-11 Parity With Works 2 and 7
The historical same-year build is compared on its 2011 evaluation rows with the Work 2 tables (`last_3`, reference 2011, year 2011): embedding spectrum, gradient coherence and gradient dimension. The prospective build is compared on its evaluation rows with the Work 7 geometry summary (reference 2011). Tolerances as in the constants. Work 2 coherence rows have no chart column; they are compared with the raw chart. Work 2 coherence is the resultant length of the log-odds gradients, so it is compared with the notebook's `raw|log_odds` resultant length, not the probability-gradient one. A log-odds row is the probability gradient times a positive factor, so after unit-normalising the two resultant lengths are equal when no row is dropped. The comparison needs the full evaluation rows, so it is skipped in a dry run that limits them.

In [12]:
parity_rows = []


def add_parity_row(source, build_name, scaling_name, metric_name, saved, measured):
    passed = bool(np.isclose(measured, saved, equal_nan=True, **FORWARD_TOLERANCE))
    parity_rows.append({"source": source, "build": build_name, "scaling": scaling_name, "metric": metric_name,
                        "saved_value": saved, "measured_value": measured,
                        "max_abs_difference": abs(measured - saved),
                        "rtol": FORWARD_TOLERANCE["rtol"], "atol": FORWARD_TOLERANCE["atol"], "passed": passed})
    return passed


def layer_eleven_value(table, section, key, metric):
    selected = table.loc[(table["section"] == section) & (table["key"] == key) & (table["metric"] == metric), "value"]
    return float(selected.iloc[0])


if EVAL_ROWS_PER_STRATUM is not None:
    print("Layer-11 parity skipped: the dry run limits the evaluation rows")
else:
    historical = "same_year_historical"
    rows = BUILD_ROWS[historical]
    subset_positions = np.flatnonzero(year[rows["evaluation"]] == REFERENCE_YEAR)
    subset_capture = captures[historical]["evaluation"]
    subset_table = lc.layer_geometry_table(
        subset_capture["target"][subset_positions], gradients_by_key[historical][subset_positions],
        subset_capture["probabilities"][subset_positions, 1], scale_rows_by_key[historical]["target"],
        feature_mean=subset_capture["feature_mean"][subset_positions],
        scale_feature_mean=scale_rows_by_key[historical]["feature_mean"])
    eleven_2011 = subset_table[(subset_table["layer"] == LAST_LAYER) & (subset_table["token"] == "target")]

    work2_start = len(parity_rows)
    spectrum = pd.read_csv(WORK2_DIR / "embedding_spectrum.csv")
    spectrum = spectrum[(spectrum["build"] == WORK2_BUILD) & (spectrum["reference"] == REFERENCE_YEAR)
                        & (spectrum["year"] == REFERENCE_YEAR)]
    axis_columns = [f"variance_share_axis_{axis}" for axis in range(1, 11)]
    for _, saved in spectrum.iterrows():
        saved_values = {"embedding_pr": saved["embedding_participation_ratio"],
                        "top1_var_share": saved["variance_share_axis_1"],
                        "top5_var_share": saved[axis_columns[:5]].sum(),
                        "top10_var_share": saved[axis_columns].sum()}
        for metric, saved_value in saved_values.items():
            add_parity_row("work2 embedding_spectrum.csv", "same_year_last_3_2011", "historical",
                           f"spectrum|{saved['chart']}|{metric}", float(saved_value),
                           layer_eleven_value(eleven_2011, "spectrum", saved["chart"], metric))

    coherence = pd.read_csv(WORK2_DIR / "coherence.csv")
    coherence = coherence[(coherence["build"] == WORK2_BUILD) & (coherence["reference"] == REFERENCE_YEAR)
                          & (coherence["panel"] == "primary") & (coherence["year"] == REFERENCE_YEAR)]
    for _, saved in coherence.iterrows():
        add_parity_row("work2 coherence.csv (log-odds resultant length)", "same_year_last_3_2011", "historical",
                       "coherence|raw|log_odds|resultant_length", float(saved["resultant_length"]),
                       layer_eleven_value(eleven_2011, "coherence", "raw|log_odds", "resultant_length"))

    dimension = pd.read_csv(WORK2_DIR / "gradient_dimension.csv")
    dimension = dimension[(dimension["build"] == WORK2_BUILD) & (dimension["reference"] == REFERENCE_YEAR)
                          & (dimension["panel"] == "primary") & (dimension["year"] == REFERENCE_YEAR)
                          & (dimension["gradient_scale"] == "probability")]
    for _, saved in dimension.iterrows():
        key_name = f"{saved['chart']}|probability"
        for metric, column in (("centered_pr", "participation_ratio_centred"),
                               ("uncentered_pr", "participation_ratio_uncentred")):
            add_parity_row("work2 gradient_dimension.csv (probability gradients)", "same_year_last_3_2011", "historical",
                           f"dimension|{key_name}|{metric}", float(saved[column]),
                           layer_eleven_value(eleven_2011, "dimension", key_name, metric))
    work2_parity = pd.DataFrame(parity_rows[work2_start:])
    record_check("work2 last_3 2011 layer-11 values match (historical same-year, 2011 rows)",
                 float(work2_parity["max_abs_difference"].max()),
                 f"{len(work2_parity)} values; allclose rtol 1e-4, atol 1e-5",
                 bool(work2_parity["passed"].all()), build="same_year_last_3_2011", scaling="historical")

    work7_start = len(parity_rows)
    work7 = pd.read_csv(PROSPECTIVE_GEOMETRY_PATH)
    work7 = work7[work7["reference_year"] == REFERENCE_YEAR].copy()
    work7["key"] = work7["key"].fillna("")
    prospective_table = geometry_tables["prospective_consistent"]
    prospective_eleven = prospective_table[(prospective_table["layer"] == LAST_LAYER)
                                           & (prospective_table["token"] == "target")][["section", "key", "metric", "value"]].copy()
    prospective_eleven["key"] = prospective_eleven["key"].fillna("")
    merged = work7.merge(prospective_eleven, on=["section", "key", "metric"], how="outer",
                         suffixes=("_saved", "_measured"), indicator=True)
    unmatched_rows = int((merged["_merge"] != "both").sum())
    for _, row in merged[merged["_merge"] == "both"].iterrows():
        add_parity_row("work7 scope_a geometry_summary.csv", "prospective_last_3_2011", "consistent",
                       f"{row['section']}|{row['key']}|{row['metric']}",
                       float(row["value_saved"]), float(row["value_measured"]))
    work7_parity = pd.DataFrame(parity_rows[work7_start:])
    record_check("work7 r=2011 layer-11 geometry rows all matched", unmatched_rows,
                 "= 0 rows present on one side only", unmatched_rows == 0,
                 build="prospective_last_3_2011", scaling="consistent")
    record_check("work7 r=2011 layer-11 geometry values match", float(work7_parity["max_abs_difference"].max()),
                 f"{len(work7_parity)} values; allclose rtol 1e-4, atol 1e-5",
                 bool(work7_parity["passed"].all()), build="prospective_last_3_2011", scaling="consistent")

    pd.DataFrame(parity_rows).to_csv(OUT_DIR / "parity_layer_11.csv", index=False)
    print(f"Work 2 values compared: {len(work2_parity)} | Work 7 values compared: {len(work7_parity)} | "
          f"Work 7 rows on one side only: {unmatched_rows}")

checks_so_far = write_checks_and_stop_on_failure()
print(f"Checks passed so far: {int(checks_so_far['passed'].sum())}/{len(checks_so_far)}")

Work 2 values compared: 13 | Work 7 values compared: 85 | Work 7 rows on one side only: 0
Checks passed so far: 53/53


## 13. Joint Patient Bootstrap Curves
One patient-level bootstrap per build over the fitting and evaluation patients (patient = `patient_id`). Each draw refits every layer's probe on the weighted fitting rows (regularization chosen once on the unweighted fitting rows) and recomputes the geometry on the weighted evaluation rows with the scales fixed. The output has estimates, 95% percentile intervals and the paired difference from layer 11.

In [13]:
curve_frames, probe_frames = [], []
for build in BUILDS:
    key = build["key"]
    rows = BUILD_ROWS[key]
    fit_capture = captures[key]["fit"]
    evaluation_capture = captures[key]["evaluation"]
    started = time.monotonic()
    fit_payload = {"patient_ids": patient_id[rows["fit"]], "y": rows["label"][rows["fit"]].astype(np.int64),
                   "target": fit_capture["target"], "feature_mean": fit_capture["feature_mean"]}
    evaluation_payload = {STRATUM: {
        "patient_ids": patient_id[rows["evaluation"]], "y": rows["label"][rows["evaluation"]].astype(np.int64),
        "probabilities": evaluation_capture["probabilities"][:, 1], "gradients": gradients_by_key[key],
        "target": evaluation_capture["target"], "feature_mean": evaluation_capture["feature_mean"]}}
    curves = lc.bootstrap_layer_curves(
        fit_payload, evaluation_payload, scale_rows_by_key[key]["target"], scale_rows_by_key[key]["feature_mean"],
        geometry_table=geometry_tables[key], draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED)
    curve_table = curves["layer_curves"]
    curve_table.insert(0, "scaling", build["scaling"])
    curve_table.insert(0, "build", build["build"])
    fit_table = curves["probe_fits"]
    fit_table.insert(0, "scaling", build["scaling"])
    fit_table.insert(0, "build", build["build"])
    curve_frames.append(curve_table)
    probe_frames.append(fit_table)
    STEP_SECONDS[f"bootstrap_{key}"] = round(time.monotonic() - started, 2)
    print(key, "| bootstrap seconds", STEP_SECONDS[f"bootstrap_{key}"])

layer_curves = pd.concat(curve_frames, ignore_index=True)
probe_fits = pd.concat(probe_frames, ignore_index=True)
layer_curves.to_csv(OUT_DIR / "layer_curves.csv", index=False)
probe_fits.to_csv(OUT_DIR / "probe_fits.csv", index=False)
infinite_estimates = int(np.isinf(layer_curves["estimate"].astype(float)).sum())
record_check("curve estimates have no infinite values", infinite_estimates,
             "no +inf or -inf among estimates (NaN counted in run record)", infinite_estimates == 0)
print(probe_fits.groupby(["build", "scaling", "token"], dropna=False)["C"].describe()[["count", "min", "max"]].to_string())

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

... (4,976 outputs left out)

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_mo

prospective_consistent | bootstrap seconds 412.51
                                                 count       min           max
build                   scaling    token                                      
prospective_last_3_2011 consistent feature_mean   12.0  0.000100      2.782559
                                   model           0.0       NaN           NaN
                                   target         12.0  0.000774     21.544347
same_year_last_3_2011   consistent feature_mean   12.0  0.359381   1291.549665
                                   model           0.0       NaN           NaN
                                   target         12.0  0.046416    166.810054
                        historical feature_mean   12.0  0.359381  10000.000000
                                   model           0.0       NaN           NaN
                                   target         12.0  0.046416    166.810054


## 14. Check Gate
Every engineering check must pass before the layer arrays and figures are written. A failed check stops the notebook here.

In [14]:
checks_table = write_checks_and_stop_on_failure()
print(checks_table[["build", "scaling", "check", "measured", "passed"]].to_string(index=False))
print(f"All {len(checks_table)} checks passed: {bool(checks_table['passed'].all())}")

                  build    scaling                                                                     check                                                         measured  passed
                    all       none                          population file sha256 equals the approved value c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7    True
                    all       none                                                  population feature count                                                              500    True
prospective_last_3_2011       none                      prospective archive roles equal reference-year roles                                                                1    True
prospective_last_3_2011       none            prospective context mask equals prospective_audit.context_mask                                                                1    True
                    all       none               same_year context, fitting and evaluation

## 15. Per-Row Arrays
Save the evaluation rows of each build (ids, years, labels, probabilities, target and feature-mean embeddings, target gradients at every layer) and the per-layer standardization scales, in one archive with keys prefixed by build, so that the curves can be recomputed without a new fit.

In [15]:
started = time.monotonic()
array_payload = {}
for build in BUILDS:
    key = build["key"]
    rows = BUILD_ROWS[key]
    evaluation_capture = captures[key]["evaluation"]
    prefix = f"{key}__"
    array_payload[prefix + "eval_row_index"] = rows["evaluation"]
    array_payload[prefix + "patient_id"] = patient_id[rows["evaluation"]].astype(str)
    array_payload[prefix + "year"] = year[rows["evaluation"]]
    array_payload[prefix + "label"] = rows["label"][rows["evaluation"]]
    array_payload[prefix + "probability"] = evaluation_capture["probabilities"][:, 1]
    array_payload[prefix + "target_embedding"] = evaluation_capture["target"]
    array_payload[prefix + "feature_mean_embedding"] = evaluation_capture["feature_mean"]
    array_payload[prefix + "target_gradient"] = gradients_by_key[key]
    array_payload[prefix + "scale_target"] = layer_scales_by_key[key]["target"]
    array_payload[prefix + "scale_feature_mean"] = layer_scales_by_key[key]["feature_mean"]
np.savez_compressed(OUT_DIR / "layer_arrays.npz", **array_payload)
STEP_SECONDS["arrays"] = round(time.monotonic() - started, 2)
print(f"layer_arrays.npz written with {len(array_payload)} arrays")

layer_arrays.npz written with 30 arrays


## 16. Figures Across Layers
One figure per metric family. Panels are the three builds; lines are the estimates with their 95% intervals; the dotted vertical line marks layer 11. The probe figure also draws the model's own AUROC on the same rows and draws.

In [16]:
started = time.monotonic()
layer_curves["layer"] = layer_curves["layer"].astype(float)


def build_curves(build):
    return layer_curves[(layer_curves["build"] == build["build"]) & (layer_curves["scaling"] == build["scaling"])]


def curve_part(curves, token, chart, metric):
    part = curves[(curves["token"] == token) & (curves["chart"] == chart) & (curves["metric"] == metric)]
    return part.sort_values("layer")


def plot_families(specs, title, file_name, y_label, baseline=False):
    figure, axes = plt.subplots(1, len(BUILDS), figsize=(4.4 * len(BUILDS), 4), sharey=True, constrained_layout=True)
    for axis, build in zip(axes, BUILDS):
        curves = build_curves(build)
        for label, token, chart, metric in specs:
            part = curve_part(curves, token, chart, metric)
            axis.plot(part["layer"], part["estimate"], marker="o", label=label)
            axis.fill_between(part["layer"], part["low"], part["high"], alpha=0.2)
        if baseline:
            model = curve_part(curves, "model", "none", "death_auroc").iloc[0]
            axis.axhline(model["estimate"], color="black", linestyle="--", label="model death AUROC")
            axis.axhspan(model["low"], model["high"], color="gray", alpha=0.15)
        axis.axvline(LAST_LAYER, color="gray", linestyle=":")
        axis.set_title(build["label"])
        axis.set_xlabel("layer")
    axes[0].set_ylabel(y_label)
    axes[0].legend(fontsize=8)
    figure.suptitle(title)
    figure.savefig(FIG_DIR / file_name, dpi=120)
    plt.close(figure)


plot_families(
    [("target token", "target", "none", "probe_auroc"), ("feature mean", "feature_mean", "none", "probe_auroc")],
    "Linear probe AUROC by layer (95% intervals)", "probe_auroc_by_layer.png", "AUROC", baseline=True)

for metric in ("embedding_pr", "top1_var_share", "top5_var_share", "top10_var_share"):
    specs = [(f"{token_label} {chart}", token, chart, metric)
             for token, token_label in (("target", "target"), ("feature_mean", "feature mean"))
             for chart in ("raw", "standardized")]
    plot_families(specs, f"{metric} by layer (95% intervals)", f"spectrum_{metric}.png", metric)

for gradient_name in ("probability", "log_odds"):
    for metric in ("resultant_length", "centered_pr", "uncentered_pr"):
        metric_key = f"{gradient_name}_{metric}"
        specs = [(chart, "target", chart, metric_key) for chart in ("raw", "standardized")]
        plot_families(specs, f"{metric_key} by layer (95% intervals)", f"gradient_{metric_key}.png", metric_key)

figure, axes = plt.subplots(1, len(BUILDS), figsize=(4.4 * len(BUILDS), 4), sharey=True, constrained_layout=True)
for axis, build in zip(axes, BUILDS):
    curves = build_curves(build)
    for family in geometry.DIRECTION_FAMILIES:
        part = curve_part(curves, "target", "raw", f"random_{family}_interior_share")
        spread = curve_part(curves, "target", "raw", f"random_{family}_interior_se")
        axis.errorbar(part["layer"], part["estimate"], yerr=spread["estimate"], marker="o", capsize=2, label=family)
    axis.axvline(LAST_LAYER, color="gray", linestyle=":")
    axis.set_title(build["label"])
    axis.set_xlabel("layer")
axes[0].set_ylabel("interior share (+/- 1 SE of directions)")
axes[0].legend(fontsize=8)
figure.suptitle("Random-direction interior share by layer (raw chart)")
figure.savefig(FIG_DIR / "random_direction_interior_share.png", dpi=120)
plt.close(figure)
print(sorted(path.name for path in FIG_DIR.glob("*.png")))
STEP_SECONDS["figures"] = round(time.monotonic() - started, 2)

['gradient_log_odds_centered_pr.png', 'gradient_log_odds_resultant_length.png', 'gradient_log_odds_uncentered_pr.png', 'gradient_probability_centered_pr.png', 'gradient_probability_resultant_length.png', 'gradient_probability_uncentered_pr.png', 'probe_auroc_by_layer.png', 'random_direction_interior_share.png', 'spectrum_embedding_pr.png', 'spectrum_top10_var_share.png', 'spectrum_top1_var_share.png', 'spectrum_top5_var_share.png']


## 17. Three-Regime Comparison With FLChain
Target token, consistent scaling: the SUS same-year and prospective builds against the FLChain curves of the FLChain notebook (`1998-2003` and the held-out `1997_eval` stratum). The historical same-year build is a dashed sensitivity line. Every line is read from the saved curve tables; no layer is selected and no rule is applied.

In [17]:
started = time.monotonic()
flchain_curves = pd.read_csv(FLCHAIN_CURVES_PATH)
flchain_curves = flchain_curves[(flchain_curves["build"] == "horizon_1095d") & (flchain_curves["scaling"] == "consistent")
                                & (flchain_curves["token"] == "target") & flchain_curves["layer"].notna()].copy()
flchain_curves["layer"] = flchain_curves["layer"].astype(float)

OVERLAY_SERIES = [
    ("SUS same-year 2011 (consistent)", "sus", "same_year_consistent", STRATUM, "-"),
    ("SUS prospective r=2011 (consistent)", "sus", "prospective_consistent", STRATUM, "-"),
    ("SUS same-year 2011 (historical, sensitivity)", "sus", "same_year_historical", STRATUM, "--"),
    ("FLChain 1998-2003", "flchain", None, "1998-2003", "-"),
    ("FLChain 1997 held-out", "flchain", None, "1997_eval", "-"),
]
OVERLAY_PANELS = [
    ("probe_auroc", "none", "probe AUROC, target token"),
    ("embedding_pr", "raw", "participation ratio, raw chart"),
    ("top1_var_share", "raw", "top-1 variance share, raw chart"),
    ("probability_resultant_length", "raw", "gradient resultant length (probability)"),
    ("probability_centered_pr", "raw", "gradient participation ratio, centred (probability)"),
]


def overlay_part(series, metric, chart):
    source, key, stratum, _ = series[1], series[2], series[3], series[4]
    if source == "sus":
        build = next(item for item in BUILDS if item["key"] == key)
        curves = layer_curves[(layer_curves["build"] == build["build"]) & (layer_curves["scaling"] == build["scaling"])
                              & (layer_curves["stratum"] == stratum)]
    else:
        curves = flchain_curves[flchain_curves["stratum"] == stratum]
    part = curves[(curves["token"] == "target") & (curves["chart"] == chart) & (curves["metric"] == metric)]
    return part.sort_values("layer")


for metric, chart, _ in OVERLAY_PANELS:
    for series in OVERLAY_SERIES:
        part = overlay_part(series, metric, chart)
        record_check(f"overlay layers complete: {series[0]}, {metric}", len(part),
                     "= 12 layers", len(part) == LAYER_COUNT)
write_checks_and_stop_on_failure()

figure, axes = plt.subplots(1, len(OVERLAY_PANELS), figsize=(4.2 * len(OVERLAY_PANELS), 4), constrained_layout=True)
for axis, (metric, chart, title) in zip(axes, OVERLAY_PANELS):
    for series in OVERLAY_SERIES:
        part = overlay_part(series, metric, chart)
        axis.plot(part["layer"], part["estimate"], series[4], marker="o", markersize=4, label=series[0])
        axis.fill_between(part["layer"], part["low"], part["high"], alpha=0.1)
    axis.axvline(LAST_LAYER, color="gray", linestyle=":")
    axis.set_title(title, fontsize=9)
    axis.set_xlabel("layer")
axes[0].legend(fontsize=7)
figure.suptitle("Target token, consistent scaling: SUS builds against FLChain (95% intervals)")
figure.savefig(FIG_DIR / "three_regime_layer_overlay.png", dpi=120)
plt.close(figure)
STEP_SECONDS["overlay"] = round(time.monotonic() - started, 2)
print("overlay written; FLChain curves sha256:", input_sha256["flchain_layer_curves.csv"])

overlay written; FLChain curves sha256: a915d4dd5c6c3633d66f9acb036659d833d62c306de62fad58613059c611169b


## 18. Run Record
Times, hardware, versions, code and input hashes, row counts, per-step seconds, peak memory and the check total. Nothing here is a conclusion.

In [18]:
checks_table = write_checks_and_stop_on_failure()
git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
git_uncommitted = subprocess.check_output(["git", "status", "--porcelain"], text=True).splitlines()
run_record = {
    "started_utc": STARTED_UTC,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "runtime_seconds": round(time.monotonic() - T0, 2),
    "step_seconds": STEP_SECONDS,
    "device": DEVICE,
    "gpu_name": None,
    "cuda_version": torch.version.cuda,
    "torch_threads": torch.get_num_threads(),
    "bootstrap_blas_thread_limit": lc.BOOTSTRAP_THREAD_LIMIT,
    "thread_environment": {name: os.environ.get(name) for name in
                           ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS")},
    "peak_gpu_memory_mib": None,
    "peak_rss_mib": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
    "git_head": git_head,
    "git_uncommitted": git_uncommitted,
    "sha256": {
        "sus_layer_compression.ipynb": probe.file_sha256("sus_layer_compression.ipynb"),
        "layer_compression.py": probe.file_sha256("layer_compression.py"),
        "flchain_probe.py": probe.file_sha256("flchain_probe.py"),
        "prospective_audit.py": probe.file_sha256("prospective_audit.py"),
        "tcav_geometry.py": probe.file_sha256("tcav_geometry.py"),
        "concept_subgroup_drift_pilot.py": probe.file_sha256("concept_subgroup_drift_pilot.py"),
    },
    "inputs_sha256": input_sha256,
    "versions": {name: metadata.version(name) for name in
                 ("torch", "drift-resilient-tabpfn", "scikit-learn", "numpy", "pandas")},
    "builds": {build["key"]: {
        "build": build["build"], "scaling": build["scaling"], "label": build["label"],
        **model_metadata[build["key"]],
        "fitting_rows": len(BUILD_ROWS[build["key"]]["fit"]),
        "fitting_deaths": int(np.sum(BUILD_ROWS[build["key"]]["label"][BUILD_ROWS[build["key"]]["fit"]])),
        "scale_rows": int(BUILD_ROWS[build["key"]]["scale_in_fit"].sum()),
        "evaluation_rows": len(BUILD_ROWS[build["key"]]["evaluation"]),
        "evaluation_deaths": int(np.sum(BUILD_ROWS[build["key"]]["label"][BUILD_ROWS[build["key"]]["evaluation"]])),
        "fd_max_abs_error_by_layer": fd_by_key[build["key"]],
        "standardization_zero_columns_target": zero_columns_by_key[build["key"]],
    } for build in BUILDS},
    "row_counts": counts.to_dict(orient="records"),
    "evaluation_rows_per_stratum_limit": EVAL_ROWS_PER_STRATUM,
    "parity_checked": EVAL_ROWS_PER_STRATUM is None,
    "bootstrap_draws": BOOTSTRAP_DRAWS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "direction_seed": DIRECTION_SEED,
    "fd_direction_seed": FD_DIRECTION_SEED,
    "fd_rows_per_layer": FD_ROWS,
    "nan_curve_estimates": int(layer_curves["estimate"].astype(float).isna().sum()),
    "checks_passed": int(checks_table["passed"].sum()),
    "checks_total": int(len(checks_table)),
}
(OUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2, default=str))
print(f"Run completed in {run_record['runtime_seconds']} s; checks passed {run_record['checks_passed']}/{run_record['checks_total']}")

Run completed in 2909.36 s; checks passed 79/79


## Descriptive reading

Written after the run (2026-11-26). Descriptive only: no layer is selected here, and no cutoff is applied. Numbers are estimate [2.5%, 97.5%] from the joint patient bootstrap (500 draws). The random-direction share is a point value with its direction-sampling SE. All checks passed (79/79). Layer-11 parity: Work 2 max abs difference 1.1e-8, Work 7 5.8e-7.

**Pooled curves (the pre-declared primary quantities).** The evaluation pools input years 2011–2013.
- **Embedding PR.** The target-token raw PR stays near 1 through layer 7 (same-year consistent 1.04 → 1.22). It then rises to 2.04 at layer 10 and 3.11 [2.94, 3.29] at layer 11. The prospective build behaves the same way, ending at 3.33 [3.23, 3.42].
- **Gradient.** R is the resultant length of the probability gradients (1 = all rows point the same way).
  - Same-year: R rises from 0.68 to 0.89 at layer 8, falls to 0.84 at layer 10, then jumps to 0.967 at layer 11.
  - Prospective: R rises steadily, 0.78 → 0.996.
  - The random isotropic interior share falls over the same layers: same-year 0.83 → 0.22 (0.48 / 0.55 at layers 9 / 10), prospective 0.67 → 0.09 (0.26 / 0.17).
- **Probe.** Comparisons use the paired difference from layer 11 within the same bootstrap draws.
  - Same-year consistent: layer 11 gives 0.834 [0.783, 0.864]; the model's own AUROC is 0.812. Layer 9 is the only earlier layer whose paired-difference interval includes 0: −0.080 [−0.166, 0.011]. All other layers have intervals below 0, layer 8 for example −0.048 [−0.188, −0.016].
  - Historical: layers 9 and 10 have intervals that include 0 (−0.036 [−0.200, 0.015] and −0.014 [−0.152, 0.013]).
  - Prospective: point values are 0.44–0.67 across layers, against the model's 0.641 [0.572, 0.703]. Every layer's paired-difference interval includes 0 (35 fitting deaths).

**The pooled embedding PR mixes input years.** The leading principal direction of the pooled rows tracks input year (`by_year/leading_direction_year_correlation.csv`):
- |r| is 0.88–0.99 through layer 9 in all three builds.
- At layer 10 it is 0.88–0.89 in the consistent builds and 0.74 in the historical build.
- At layer 11 it is 0.08 for same-year and 0.84 for prospective.

In the consistent builds that direction holds 69–98% of the variance through layer 10. So their near-1 pooled PR at early and middle layers mostly reflects the spread between years, not a narrow within-year representation. In the historical build it holds only 33–40% at layers 8–10. The per-year split, done after the run and outside the pre-declared plan, is in `layer_compression_by_year.ipynb` (`by_year/`). Within each input year:
- **Embedding PR.** Raw PR at layer 9 is higher than at layer 11 in all 9 build × year cells. The paired difference interval excludes 0 in 8 of the 9. By build:
  - same-year consistent: 4.0–4.4 vs 2.3–2.9 (+1.1 to +2.0);
  - historical: 4.4–4.6 vs 2.2–2.3;
  - prospective: 3.1–3.4 vs 2.6–2.9 (+0.3 to +0.6).

  The standardized chart shows the same for same-year (+0.7 to +1.8). For prospective it shows no difference (all three intervals include 0).
- **Gradient.** Gradients at layers 9–10 are less concentrated than at layer 11 in every cell:
  - same-year: R 0.84–0.89 vs 0.96–0.97; centred PR 1.25–1.38 vs about 1.1; random isotropic share 0.42–0.55 vs 0.21–0.23;
  - prospective: R 0.978–0.984 vs 0.997; random share 0.15–0.20 vs 0.08. The gap is much smaller than in the same-year build.
- **Probe.** Per-year probe AUROC was not computed, because the fitting-row arrays were not saved. The pooled probe values above are the only probe evidence.

**What this shows, descriptively.**
- Gradient concentration and random-direction TCAV saturation grow over layers 0–10, though not monotonically. The same-year builds dip between layers 8 and 10. Then there is a final step at layer 11, largest in the same-year builds.
- Within an input year, layer 9 holds a target-token representation with more effective dimensions than layer 11: the point estimate is higher in all 9 build × year cells, and the interval excludes 0 in 8 (not prospective 2011). Layer 10 shows the same in 8 of 9 cells (again not prospective 2011).
- Pooled probe AUROC at layers 9–10 is 0.69–0.82 in the same-year builds, against 0.83 at layer 11. Only the intervals listed above include 0.
- Choosing a layer for any follow-up is left to the user.

**Caveats.**
- **Partial derivative.** Gradients are partial derivatives w.r.t. the test-row target token, holding the other tokens of that layer fixed. They are not the full Jacobian through attention.
- **Mixed target token at layer 0.** The layer-0 target token has already been mixed by feature attention. Every token lies on a sphere of norm sqrt(192) (LayerNorm without affine).
- **Record truncation.** Same-year probe and model AUROC partly reflect record truncation; record end alone gives AUROC 0.84 in earlier work. The prospective target removes that.
- **Small fitting sets.** Fitting sets are small (same-year 789 rows / 81 deaths; prospective 696 / 35), so probe intervals are wide. C sits at the edge of the CV grid in 2 of 72 probe fits (`probe_fits.csv`, `C_at_grid_edge`).
- **Context overlap.** Earlier-year rows of about half the evaluation patients are in the model context (same-year 252 of 454; prospective 210 of 391). This is the inherited design.
- **Post-death rows.** The same-year evaluation includes 14 post-death rows (Work 2 definition).
- **Padded tokens.** The feature-mean control averages all T−1 feature tokens, including padded ones (T = 121 same-year, 104 prospective).
- **Scaling sensitivity.** Historical scaling changes the pooled mid-layer curves strongly (pooled raw PR 4.7 at layer 9 vs 1.7 consistent). The within-year curves are close between the two scalings.